# DE300 Lab3: AWS Setup
AWS (Amazon Web Services) is a cloud computing platform provided by Amazon. It offers a wide range of services like computing power, storage, networking, machine learning, and databases. Here are some functions of AWS that we may use in this course:

- **EC2 (Elastic Compute Cloud)**: Virtual servers in the cloud.
- **S3 (Simple Storage Service)**: Object storage for data, backups, static websites, etc.
- **RDS**: Managed relational databases (e.g., MySQL, PostgreSQL).
- **VPC (Virtual Private Cloud)** : VPC (Virtual Private Cloud): Create isolated networks in the cloud.
- **EMR (Elastic MapReduce)**: : Hadoop/Spark clusters

Today we are going to create EC2 instances on AWS. EC2 (Elastic Compute Cloud) is a resizable compute capacity in the cloud. You can think of it as a virtual machine (VM) running on Amazon’s infrastructure, where
- You choose the OS (e.g., Ubuntu, Amazon Linux).
- You can install software, run scripts, host servers.
- You can SSH into it like any local server.

Most Amazon Machine Images (AMIs) are minimal (CLI-only). If you want a Windows or Linux desktop with full GUI out-of-the-box, try Amazon WorkSpaces instead of EC2.

## 1. Create an EC2 instance on AWS and SSH your instance

1. Login at [AWS](https://nu-sso.awsapps.com/start/) (https://nu-sso.awsapps.com/start/)
2. Navigate to `mse-tl-dataeng300-EMR` → `EC2` → `Launch instance`. Choose `Launch instance from template`, then choose `de300-t2.medium`. You don't have to modify most of the instance parameters here:
   - **Application and OS Images**: Choose Ubuntu (AWS Linux is also an option).
   - **Instance type**: `t2.medium` is recommended.
   - **Key pair (IMPORTANT)**: Select `create new key pair`. Enter a preferred file name and download the key file. 
   - **Network Settings **: Click `EDIT`. Select `VPC ID: vpc-0f4855643d7cd7d56 (MWAAEnvironment)` The series
’-0f4855643d7cd7d56’ you find in your case can be different, but make sure to choose the one with ‘MWAAEnvironment’. Make sure ‘Auto Assign Public IP’ is ‘enabled’.
   - **Security Group Name**: Choose a name for easier reference.
   - **Configure Storage**: Choose `30 GiB` with `gp3`.
1. Launch the instance and wait for the state to switch to `running`.
2. **SSH your instance:**
   - Click the instance ID and select `connect` > `SSH client`.
   - Move to the key path folder and follow the instructions provided.
   - Run the `ssh -i` command and type `yes` when prompted to create a footprint.

## 2. Create a s3 bucket
Amazon S3 (Simple Storage Service) is AWS’s object storage service. We often save files such as datasets, notebooks, model outputs, logs, and others here because
- it's easier to organize
- it' convinient to share the files acrros AWS services
- the files are kept even after an EC2 instance is stopped or removed

S3 works with two main concepts:
- **bucket**: a container for files,
- **object**: actual file stored in that bucket + its metadata

A typical workflow
1. Create a bucket.
2. Upload files (objects) into the bucket.
3. List, download, or copy objects when needed.
4. Organize files by prefixes such as data/, output/, or logs/.
5. Delete objects you no longer need.

### Create a bucket in the browser
1. Sign in to the AWS Management Console and open Amazon S3.
2. In the left panel, choose General purpose buckets.
3. Click Create bucket.
4. Enter a bucket name. The name must be globally unique, so a common choice is something like `netid-course-lab3` or `lastname-firstname-lab3`.
5. Choose the AWS Region.
6. Leave the default settings unless your lab requires something different. In particular, it is usually best to keep Block all public access enabled.
7. Click Create bucket.

## 3. Introduction to Docker
Docker is a platform that allows developers to create, deploy, and run applications in containers. Containers are lightweight, portable, and ensure that the application runs consistently across different environments. In data engineering, Docker is commonly used for reproducibility, managing data pipelines, and running ETL jobs.

### Understanding Docker Concepts
- **Image:** A lightweight, standalone, executable package that includes everything needed to run a piece of software.
- **Container:** A running instance of a Docker image.
- **Dockerfile:** A script containing a series of commands used to create a Docker image.

### Install Docker
1. Update package index:
   ```bash
   sudo dnf update -y
   sudo dnf install -y docker
   ```
2. Enable Docker service:
   ```bash
   sudo systemctl enable docker
   sudo systemctl start docker
   ```
3. Verify installation:
   ```bash
   docker --version
   sudo docker run hello-world
   ```

## 4. Run Jupyter Notebook on EC2
### Restart the SSH session
We need to restart the SSH session, so first press
```
exit
```
in terminal to terminate the session. Then restart the instance by
```
ssh -i YOUR_KEY_PATH -L 8888:localhost:8888 EC2_IP_ADDRESS
```
Note that we add the option '-L 8888:localhost:8888' to create a port tunnel between EC2 8888 and your local machine 8888.

### Set Docker permissions
By using the command
```
sudo chmod 666 /var/run/docker.sock    
```

## Create your first Docker(Jupyter Notebook) Image and run the Docker Container with this image
(1) Under your course directory (on your EC2 instances), create a folder `{yourname}-image` (you can choose your preferred name). 
```
mkdir {yourname}-image
cd {yourname}-image
```

In this folder, create file with name Dockerfile that contains the following txt:

you can use vim to create such file
```
vim Dockerfile
```

```sh
# Use the Jupyter Data Science Notebook as the base image
FROM jupyter/datascience-notebook

# Set the working directory to /home/jovyan - the default for Jupyter images
WORKDIR /home/jovyan

EXPOSE 8888

# Avoid prompts from apt during the build process
ENV DEBIAN_FRONTEND=noninteractive

COPY . .

# Install the latest AWS CLI version 2
USER root

# Update and install necessary packages
RUN apt-get update && apt-get install -y \
    wget \
    curl \
    unzip \
    bzip2 \
    git \
    && rm -rf /var/lib/apt/lists/*

# Install the AWS CLI tools to your image
RUN curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip" && \
    unzip awscliv2.zip && \
    ./aws/install && \
    rm -rf awscliv2.zip ./aws
USER jovyan


# Start the Jupyter Notebook server when the container launches
# The base image already configures Jupyter to run on start, but you can customize
# startup options here if needed.
CMD ["start-notebook.sh", "--NotebookApp.token=''", "--NotebookApp.password=''", "--allow-root"]

```

After copy and paste, press `esc`, then input `:wq` to save and quit.


(2) Create your image, run 
```
docker build -t my_jupyter .
```
After running the above command, you can check the image is created by running 'docker images'. Building the myjupyter image first time will take some time, since this image is built from the base image 'jupyter/datascience-notebook', which is about 6GB and it takes time to download from the 'docker hub' (this is like github repo to git, search this!). So when you 'docker images', you can actuallly see two images, one is 'jupyter/datascience-notebook' as the base image, the other is the one you created with the name 'myjupyter'.

(3) Create your SAMPLE_PROJ_PATH. Run the container in your EC2 instance with 'my_jupytr' image:
```  
docker run -p 8888:8888 -v SAMPLE_PROJ_PATH:/home/jovyan/ my_jupyter 

```

In the above command, the option -p sets up the port tunnel: the container's 8888 tunnel, which the jupyter notebook application occupies, sync with your instance 8888 port. The option '-v SAMPLE_PROJ_PATH:/home/jovyan/' tells the container to sync your working directory 'SAMPLE_PROJ_PATH'.

You can check your container by command 'docker ps -a', which will show all your containers. Option '-a' makes it show both running and stopped containers.

Search the following command and see what they do:
```

docker start container_id
docker stop container_id
docker logs container_id

docker rm container_id
docker rmi image_id
```

## Lab Assignment 3
Run the provided jupyter notebook lab_assignment3.jpynb on EC2, and submit the notebook with oputput.